# 🌿 Crop Disease Model — Retrain with 39th Background Class
Adds a `background` class to your existing 38-class disease model.
After this the model outputs `background` for faces, hands, rooms etc.

**Runtime**: Colab T4 GPU — ~30 minutes  
**Steps**: Run all cells top to bottom

In [1]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np
import os, re, shutil, random
from pathlib import Path

print(f'TF version : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

if not tf.config.list_physical_devices('GPU'):
    print('\n⚠ No GPU! Runtime → Change runtime type → T4 GPU')

TF version : 2.19.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ── Cell 2: Kaggle auth + download datasets ────────────────────────────────
!pip install -q kaggle

from google.colab import files
print('Upload your kaggle.json:')
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# PlantVillage — 38 disease classes (leaf images)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/plantvillage --unzip -q
print('PlantVillage downloaded ✓')

# ImageNet Mini — 1000 diverse classes for background
!kaggle datasets download -d ifigotin/imagenetmini-1000 -p /content/imagenet --unzip -q
print('ImageNet Mini downloaded ✓')

# CelebA — dedicated face images (most common false positive)
!kaggle datasets download -d jessicali9530/celeba-dataset -p /content/faces --unzip -q
print('CelebA faces downloaded ✓')

Upload your kaggle.json:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
PlantVillage downloaded ✓
Dataset URL: https://www.kaggle.com/datasets/ifigotin/imagenetmini-1000
License(s): unknown
ImageNet Mini downloaded ✓
Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
CelebA faces downloaded ✓


In [3]:
# ── Cell 3: Upload your existing disease model ─────────────────────────────
from google.colab import files
print('Upload your crop_disease_mobilenet.h5:')
uploaded = files.upload()
print('Model uploaded ✓')

Upload your crop_disease_mobilenet.h5:


Saving crop_disease_mobilenet.h5 to crop_disease_mobilenet.h5
Model uploaded ✓


In [4]:
# ── Cell 4: Find PlantVillage class folders ────────────────────────────────
def collect(root, exts={'.jpg','.jpeg','.png','.JPG','.JPEG','.PNG'}):
    return [f for f in Path(root).rglob('*') if f.suffix in exts]

def normalise(name):
    n = name.lower()
    n = re.sub(r'[_,\-\(\)\s\.]+', '', n)
    n = re.sub(r'___+|__+', '', n)
    return n

# Find all image-containing folders
pv_root    = Path('/content/plantvillage')
candidates = [d for d in pv_root.rglob('*')
              if d.is_dir() and
              any(f.suffix.lower() in {'.jpg','.jpeg','.png'}
              for f in d.iterdir() if f.is_file())]

# Deduplicate — prefer triple-underscore naming
seen = {}
for cls_dir in sorted(candidates):
    key = normalise(cls_dir.name)
    if key not in seen:
        seen[key] = cls_dir
    else:
        if cls_dir.name.count('___') > seen[key].name.count('___'):
            seen[key] = cls_dir
        elif ',' in seen[key].name and ',' not in cls_dir.name:
            seen[key] = cls_dir

unique_classes = list(seen.values())
print(f'PlantVillage unique classes: {len(unique_classes)}  (should be 38)')
for d in sorted(unique_classes, key=lambda x: x.name):
    print(f'  {d.name}: {len(collect(d))} images')

PlantVillage unique classes: 38  (should be 38)
  Apple___Apple_scab: 630 images
  Apple___Black_rot: 621 images
  Apple___Cedar_apple_rust: 275 images
  Apple___healthy: 1645 images
  Blueberry___healthy: 1502 images
  Cherry_(including_sour)___Powdery_mildew: 1052 images
  Cherry_(including_sour)___healthy: 854 images
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513 images
  Corn_(maize)___Common_rust_: 1192 images
  Corn_(maize)___Northern_Leaf_Blight: 985 images
  Corn_(maize)___healthy: 1162 images
  Grape___Black_rot: 1180 images
  Grape___Esca_(Black_Measles): 1383 images
  Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1076 images
  Grape___healthy: 423 images
  Orange___Haunglongbing_(Citrus_greening): 5507 images
  Peach___Bacterial_spot: 2297 images
  Peach___healthy: 360 images
  Pepper,_bell___Bacterial_spot: 997 images
  Pepper,_bell___healthy: 1478 images
  Potato___Early_blight: 1000 images
  Potato___Late_blight: 1000 images
  Potato___healthy: 152 images
  Raspb

In [5]:
# ── Cell 5: Build dataset — 38 disease classes + 1 background ─────────────
DATASET_DIR = Path('/content/disease_39')
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ['train', 'val']:
    (DATASET_DIR / split).mkdir(parents=True, exist_ok=True)

TRAIN_RATIO = 0.85
total_imgs  = 0

# ── 38 disease classes from PlantVillage ──────────────────────────────────
for cls_dir in sorted(unique_classes):
    imgs = collect(cls_dir)
    random.seed(42)
    random.shuffle(imgs)
    n_train = int(len(imgs) * TRAIN_RATIO)

    for split, subset in [('train', imgs[:n_train]), ('val', imgs[n_train:])]:
        dst = DATASET_DIR / split / cls_dir.name
        dst.mkdir(parents=True, exist_ok=True)
        for i, f in enumerate(subset):
            shutil.copy(f, dst / f'{i}{f.suffix}')
    total_imgs += len(imgs)

print(f'Disease classes copied ✓  ({total_imgs} images)')

# ── 39th class: background ────────────────────────────────────────────────
# Mix of ImageNet (objects/scenes) + CelebA (faces)
# Faces are the main false positive so weight them more
all_bg = []

imagenet_imgs = collect('/content/imagenet')
random.shuffle(imagenet_imgs)
all_bg += imagenet_imgs[:1200]   # 1200 diverse objects/scenes

face_imgs = collect('/content/faces')
random.shuffle(face_imgs)
all_bg += face_imgs[:800]        # 800 faces — most common false positive

random.shuffle(all_bg)
n_train = int(len(all_bg) * TRAIN_RATIO)

for split, subset in [('train', all_bg[:n_train]), ('val', all_bg[n_train:])]:
    dst = DATASET_DIR / split / 'background'
    dst.mkdir(parents=True, exist_ok=True)
    for i, f in enumerate(subset):
        shutil.copy(f, dst / f'{i}{f.suffix}')

print(f'Background class: {len(all_bg)} images  ({n_train} train, {len(all_bg)-n_train} val)')

total_classes = len(list((DATASET_DIR/'train').iterdir()))
print(f'\nTotal classes : {total_classes}  (should be 39)')
print(f'Total images  : {total_imgs + len(all_bg)}')

Disease classes copied ✓  (54305 images)
Background class: 2000 images  (1700 train, 300 val)

Total classes : 39  (should be 39)
Total images  : 56305


In [6]:
# ── Cell 6: Data pipelines ─────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'train',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=True, seed=42,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'val',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False,
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
BG_IDX      = class_names.index('background')

print(f'Classes : {NUM_CLASSES}')
print(f'background index : {BG_IDX}  ← update BG_CLASS_IDX in live_detection.py')
print(f'Class list: {class_names}')

# Save updated class names
with open('/content/class_names_39.txt', 'w') as f:
    f.write('\n'.join(class_names))
print('class_names_39.txt saved ✓')

# Preprocessing — MobileNetV2 normalisation
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

def preprocess(x, y):
    return mobilenet_preprocess(tf.cast(x, tf.float32)), y

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

train_ds = (train_ds
    .map(lambda x,y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

val_ds = (val_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

print('Data pipelines ready ✓')

Found 47843 files belonging to 39 classes.
Found 8462 files belonging to 39 classes.
Classes : 39
background index : 38  ← update BG_CLASS_IDX in live_detection.py
Class list: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_s

In [7]:
# Quick check — what layers does your uploaded model actually have?
import h5py
with h5py.File('crop_disease_mobilenet.h5', 'r') as f:
    print("Top-level keys:", list(f.keys()))
    if 'model_weights' in f:
        print("Layer names:", list(f['model_weights'].keys())[:10])

Top-level keys: ['model_weights', 'optimizer_weights']
Layer names: ['Conv1', 'Conv1_relu', 'Conv_1', 'Conv_1_bn', 'batch_normalization', 'block_10_depthwise', 'block_10_depthwise_BN', 'block_10_depthwise_relu', 'block_10_expand', 'block_10_expand_BN']


In [9]:
# Run this before Cell 7 — check exact head structure
import h5py
with h5py.File('crop_disease_mobilenet.h5', 'r') as f:
    weights = f['model_weights']
    all_layers = list(weights.keys())
    print(f'Total layers: {len(all_layers)}')
    print('Last 10 layers:')
    for l in all_layers[-10:]:
        print(f'  {l}')

Total layers: 160
Last 10 layers:
  dropout
  expanded_conv_depthwise
  expanded_conv_depthwise_BN
  expanded_conv_depthwise_relu
  expanded_conv_project
  expanded_conv_project_BN
  global_average_pooling2d
  input_1
  out_relu
  top_level_model_weights


In [10]:
# ── Cell 7: Rebuild exact architecture + load weights ─────────────────────
import h5py

# Rebuild identical architecture to original model
base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None,    # no ImageNet weights — loading from .h5 below
)

x       = base.output
x       = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling2d')(x)
x       = tf.keras.layers.Dropout(0.3, name='dropout')(x)
x       = tf.keras.layers.Dense(38, activation='softmax', name='predictions')(x)
old_model = tf.keras.Model(inputs=base.input, outputs=x)

print(f'Architecture rebuilt ✓  layers: {len(old_model.layers)}')
print(f'Output shape: {old_model.output_shape}')

# Load weights by name — matches layer names from h5 file
old_model.load_weights('crop_disease_mobilenet.h5', by_name=True, skip_mismatch=True)
print('Weights loaded ✓')

# Verify weights loaded correctly — check a known layer
w = old_model.get_layer('Conv1').get_weights()
print(f'Conv1 weight shape: {w[0].shape}  (should not be all zeros)')

# ── Build new 39-class model ───────────────────────────────────────────────
x       = old_model.layers[-2].output   # after dropout, before Dense
x       = tf.keras.layers.Dense(
    NUM_CLASSES, activation='softmax', name='predictions_39'
)(x)
model = tf.keras.Model(inputs=old_model.input, outputs=x)

# Freeze all except new head
for layer in model.layers[:-1]:
    layer.trainable = False

print(f'Trainable layers : {sum(1 for l in model.layers if l.trainable)} / {len(model.layers)}')
print(f'Output shape     : {model.output_shape}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
print('Model ready ✓')

Architecture rebuilt ✓  layers: 157
Output shape: (None, 38)
Weights loaded ✓
Conv1 weight shape: (3, 3, 3, 32)  (should not be all zeros)
Trainable layers : 1 / 157
Output shape     : (None, 39)
Model ready ✓


In [ ]:
# ── Cell 8: Phase 1 — train new head only (10 epochs) ─────────────────────
os.makedirs('/content/checkpoints', exist_ok=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=2, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/checkpoints/phase1_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print('Phase 1 — training new 39-class head …')
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=10, callbacks=callbacks,
)
print(f'\nPhase 1 best val accuracy: {max(history1.history["val_accuracy"])*100:.2f}%')

Phase 1 — training new 39-class head …
Epoch 1/10
1496/1496 ━━━━━━━━━━━━━━━━━━━━ 0s 410ms/step - accuracy: 0.7344 - loss: 1.0082
Epoch 1: val_accuracy improved from None to 0.92366, saving model to /content/checkpoints/phase1_best.h5



Epoch 1: finished saving model to /content/checkpoints/phase1_best.h5
1496/1496 ━━━━━━━━━━━━━━━━━━━━ 659s 427ms/step - accuracy: 0.8407 - loss: 0.5631 - val_accuracy: 0.9237 - val_loss: 0.2511 - learning_rate: 0.0010
Epoch 2/10
1495/1496 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.9091 - loss: 0.2853
Epoch 2: val_accuracy improved from 0.92366 to 0.93429, saving model to /content/checkpoints/phase1_best.h5



Epoch 2: finished saving model to /content/checkpoints/phase1_best.h5
1496/1496 ━━━━━━━━━━━━━━━━━━━━ 519s 347ms/step - accuracy: 0.9128 - loss: 0.2760 - val_accuracy: 0.9343 - val_loss: 0.2051 - learning_rate: 0.0010
Epoch 3/10
1495/1496 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step - accuracy: 0.9209 - loss: 0.2483
Epoch 3: val_accuracy improved from 0.93429 to 0.93938, saving model to /content/checkpoints/phase1_best.h5



Epoch 3: finished saving model to /content/checkpoints/phase1_best.h5
1496/1496 ━━━━━━━━━━━━━━━━━━━━ 550s 339ms/step - accuracy: 0.9219 - loss: 0.2440 - val_accuracy: 0.9394 - val_loss: 0.1851 - learning_rate: 0.0010
Epoch 4/10
 401/1496 ━━━━━━━━━━━━━━━━━━━━ 5:51 321ms/step - accuracy: 0.9271 - loss: 0.2202

In [ ]:
# ── Cell 9: Phase 2 — fine-tune top layers ────────────────────────────────
# Unfreeze top 30 layers for deeper adaptation
for layer in model.layers[-30:]:
    layer.trainable = True

unfrozen = sum(1 for l in model.layers if l.trainable)
print(f'Unfrozen layers: {unfrozen} / {len(model.layers)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/checkpoints/phase2_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print('Phase 2 — fine-tuning top 30 layers …')
history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=15, callbacks=callbacks2,
)
print(f'\nPhase 2 best val accuracy: {max(history2.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ── Cell 10: Evaluate ─────────────────────────────────────────────────────
loss, acc = model.evaluate(val_ds, verbose=1)
print(f'\nFinal val accuracy : {acc*100:.2f}%')
print(f'Final val loss     : {loss:.4f}')

# Per-class accuracy — check background class specifically
print('\nChecking background class accuracy …')
y_true_all, y_pred_all = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true_all.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_all.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)

bg_mask = y_true == BG_IDX
bg_acc  = (y_pred[bg_mask] == BG_IDX).mean() if bg_mask.any() else 0
print(f'Background class accuracy : {bg_acc*100:.1f}%  (higher = fewer false positives)')

# Show any disease classes that dropped below 85%
print('\nDisease class accuracy:')
low = []
for i, cls in enumerate(class_names):
    if cls == 'background':
        continue
    mask = y_true == i
    if not mask.any():
        continue
    a = (y_pred[mask] == i).mean()
    if a < 0.85:
        low.append((cls, a))

if low:
    print('⚠ Classes below 85%:')
    for c, a in sorted(low, key=lambda x: x[1]):
        print(f'  {c}: {a*100:.1f}%')
else:
    print('✓ All disease classes above 85%')

if acc >= 0.90 and bg_acc >= 0.80:
    print('\n✓ Good — safe to save')
else:
    print('\n⚠ Consider more epochs before saving')

In [ ]:
# ── Cell 11: Save model + download ────────────────────────────────────────
# Save as .h5 — compatible with your local TF 2.13
model.save('/content/crop_disease_mobilenet_39.h5')
print('Model saved ✓')

size_mb = os.path.getsize('/content/crop_disease_mobilenet_39.h5') / (1024*1024)
print(f'File size: {size_mb:.1f} MB')

from google.colab import files
files.download('/content/crop_disease_mobilenet_39.h5')
files.download('/content/class_names_39.txt')

print('\nDownloaded:')
print('  crop_disease_mobilenet_39.h5  → rename to crop_disease_mobilenet.h5')
print('                                   replace in models/')
print('  class_names_39.txt            → rename to class_names.txt')
print('                                   replace in cv_app/')
print(f'\nBG_CLASS_IDX = {BG_IDX}  ← update this in live_detection.py')